In [1]:
%pip install pandas tqdm

Note: you may need to restart the kernel to use updated packages.


In [7]:
%pip install transformers tqdm

  Using cached transformers-5.5.4-py3-none-any.whl.metadata (32 kB)
  Using cached huggingface_hub-1.11.0-py3-none-any.whl.metadata (14 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached safetensors-0.7.0-cp38-abi3-win_amd64.whl.metadata (4.2 kB)
  Using cached hf_xet-1.4.3-cp37-abi3-win_amd64.whl.metadata (4.9 kB)
Using cached transformers-5.5.4-py3-none-any.whl (10.2 MB)
Using cached huggingface_hub-1.11.0-py3-none-any.whl (645 kB)
Using cached hf_xet-1.4.3-cp37-abi3-win_amd64.whl (3.7 MB)
Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl (2.7 MB)
Using cached safetensors-0.7.0-cp38-abi3-win_amd64.whl (341 kB)

   -- -------------------------------------  1/14 [safetensors]
   -- -------------------------------------  1/14 [safetensors]
   ----- ----------------------------------  2/14 [regex]
   ----- ----------------------------------  2/14 [regex]
   ----------- ----------------------------  4/14 [hf-xet]
   -------------- ----------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.5.1 requires sympy==1.13.1; python_version >= "3.9", but you have sympy 1.14.0 which is incompatible.


In [1]:
import pandas as pd
import numpy as np
import os
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

In [2]:
df = pd.read_csv('btcusd_1-min_data.csv')

df = df.rename(columns={'Timestamp': 'date'})

#controllo valori nulli iniziali
print(df.isnull().sum())

#trasformo formato timestamp in date time
df['date'] = pd.to_datetime(df['date'], unit='s')
df.set_index('date', inplace=True)

date      0
Open      0
High      0
Low       0
Close     0
Volume    0
dtype: int64


In [3]:
#AGGREGAZIONE ORARIA
df_hourly = df.resample('h').agg({
    'Open': 'first',
    'High': 'max',
    'Low': 'min',
    'Close': 'last',
    'Volume': 'sum'
})

#verifico se ci sono buchi di ore intere nel dataset
ore_vuote = df_hourly['Close'].isnull().sum()
print(f"Ore totali generate: {len(df_hourly)}")
print(f"Ore intere senza alcun dato (NaN): {ore_vuote}")

Ore totali generate: 124239
Ore intere senza alcun dato (NaN): 18


In [5]:
#le ore vuote potrebbero trovarsi al di fuori dell'intervallo di tempo considerato
#filtro per coerenza temporale con il dataset del sentiment
inizio = '2019-01-01 00:00:00'
fine = '2019-11-23 15:00:00'
df_final = df_hourly.loc[inizio : fine]

print(f"Dataset finale aggregato per ORA ({inizio} - {fine}):")
print(df_final.head())
print(df_final.tail())
print(f"Righe totali (ore effettive) nel periodo selezionato: {len(df_final)}")

Dataset finale aggregato per ORA (2019-01-01 00:00:00 - 2019-11-23 15:00:00):
                        Open     High      Low    Close      Volume
date                                                               
2019-01-01 00:00:00  3750.62  3752.01  3630.00  3690.91  603.089248
2019-01-01 01:00:00  3685.65  3690.78  3663.13  3681.57  142.961312
2019-01-01 02:00:00  3681.00  3685.74  3660.00  3678.00  175.021283
2019-01-01 03:00:00  3676.06  3686.45  3656.60  3677.91  196.853708
2019-01-01 04:00:00  3684.68  3697.35  3679.98  3693.30  169.208314
                        Open     High      Low    Close      Volume
date                                                               
2019-11-23 11:00:00  7218.56  7239.36  7191.10  7228.28   91.823203
2019-11-23 12:00:00  7227.53  7250.82  7169.58  7189.87  167.779252
2019-11-23 13:00:00  7189.87  7229.09  7132.95  7156.75  271.558390
2019-11-23 14:00:00  7156.75  7183.52  7102.16  7179.14  396.748508
2019-11-23 15:00:00  7178.88  7214.84 

In [6]:
#salvo il dataset con i prezzi di btc
df_final.to_csv('btcusd_2019_orario.csv')

Ora lavoro sul dataset contenente i tweet.
Il dataset è formato da oltre 39 milioni di tweet dal 2013 al 2023.
Per diminuire il rumore e gestire i dati:
1) escludo tutti i tweet precedenti alla data 2018-01-02 (data di inizio di un secondo dataset)
2) non faccio leggere tutto il dataset per intero ma lo divido in chunk (da 100k tweet ciascuno)
3) escludo i tweet che hanno valore pari a 0 in anche uno solo tra numero like, numero commenti e numero retweet

Se il numero di osservazioni rimane elevato si procederà con un campionamento casuale tra i dati rimanenti

In [3]:
input_file = 'dataset_sin_spam.csv'    #carico il dataset
output_file = 'dataset_filtrato.csv'   #creo una variabile dove inserire i dati filtrati
chunk_size = 100000                    #verranno letti 100k tweet per volta

#Escludo tutti i tweet precedenti alla data selezionata
data_inizio = '2018-01-02'

#Seleziono le colonne d'interesse per la sentiment analysis
colonne_da_caricare = ['date', 'n_replies', 'n_likes', 'n_retweets', 'text']

total_original = 0   #numero di tweet originali (pre filtraggio)
total_filtered = 0   #numero di tweet filtrati

try:
    reader = pd.read_csv(
        input_file, 
        chunksize=chunk_size,        #100k tweet per ogni chunk
        usecols=colonne_da_caricare, #legge solo le colonne selezionate
        sep=';',                     #separatore delle colonne
        on_bad_lines='skip',         #l'iterazione non si interrompe se pandas "scopre" una colonna di troppo in qualche osservazione
        engine='c', 
        low_memory=False             #forza pandas a non cercare di indovinare il tipo di colonna basandosi su un numero limitato di osservazioni
    )
    
    first_chunk = True  #specifica che è il primo blocco in modo tale da inserire i nomi delle colonne
    
    for chunk in tqdm(reader, desc="Progresso"):
        total_original += len(chunk)  #aggiorna il numero di tweet originali
        
        #Aggiungo il filtro per l'engagement (escludo i tweet che hanno valore pari a 0 in anche uno solo tra numero like, commenti e retweet)
        tweet_validi = (
            (chunk['date'] >= data_inizio) & 
            (chunk['n_replies'] >= 1) & 
            (chunk['n_likes'] >= 1) & 
            (chunk['n_retweets'] >= 1)
        )
        
        filtered_chunk = chunk[tweet_validi].dropna(subset=['text'])  #rimuove le osservazioni con valore NA nella colonna text
        total_filtered += len(filtered_chunk)  #aggiorna il numero di tweet originali
        
        #Creo dataset filtrato
        filtered_chunk.to_csv(output_file, mode='a', index=False, header=first_chunk) #mode=a append / index=false forza a non creare una colonna index
        first_chunk = False #specifica che non è il primo chunk in modo tale da non inserire nuovamente nomi colonne e di unire al chunk precedente


    print(f"Tweet analizzati: {total_original:,}")
    print(f"Tweet salvati: {total_filtered:,}")

except Exception as e:
    print(f"Si è verificato un errore durante l'elaborazione: {e}")  #per esplicitare eventuali errori

Progresso: 392it [04:39,  1.40it/s]

Tweet analizzati: 39,162,387
Tweet salvati: 510,073


Per la fase di classificazione del sentiment, si è optato per l'impiego del modello CryptoBERT, un'architettura transformer pre-addestrata specificamente su dataset provenienti dal social network Twitter e inerenti al mercato dei cripto-asset. Tale scelta è stata dettata dalla peculiare natura del linguaggio finanziario digitale, caratterizzato da un elevato tasso di termini "specialistici" del settore (slang crypto) che i modelli linguistici generalisti potrebbero non essere in grado di interpretare correttamente.

Prima di procedere imposto l'analisi sulla GPU per una maggiore potenza computazionale.

In [5]:
#VOGLIO VEDERE SE PER ADDESTRARE IL MODELLO UTILIZZO LA GPU 
import torch
import sys

# 1. Verifica dove sta guardando questo specifico Notebook
print(f"Percorso Python in uso: {sys.executable}")

# 2. Controllo disponibilità CUDA
cuda_disponibile = torch.cuda.is_available()
print(f"CUDA è disponibile? {cuda_disponibile}")

if cuda_disponibile:
    # 3. Informazioni sulla scheda video
    print(f"ID GPU attuale: {torch.cuda.current_device()}")
    print(f"Nome GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memoria totale: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("\n--- DIAGNOSI ERRORE ---")

Percorso Python in uso: C:\Users\valal\anaconda3\envs\torch_gpu\python.exe
CUDA è disponibile? True
ID GPU attuale: 0
Nome GPU: NVIDIA GeForce RTX 4070 Laptop GPU
Memoria totale: 8.59 GB


In [15]:
device = 0 if torch.cuda.is_available() else -1   #device = 0 dice alla pipeline di spostare tutto il modello e i dati sulla memoria video
                                                  #device = -1 dice alla pipeline di tenere tutto sulla memoria RAM normale.

#Scarico il modello
CryptoBERT = "ElKulako/cryptobert"

print(f"Caricamento di {CryptoBERT} sulla GPU")

tokenizer = AutoTokenizer.from_pretrained(CryptoBERT) 
#tokenizer: 1) scompone ogni tweet in token, assegna un id numerico univoco basato sul dizionario con cui CryptoBERT è stato addestrato
# e uniforma la lunghezza dei tweet (aggiungendo zeri o tagliando il testo se supera i 128 caratteri)
model = AutoModelForSequenceClassification.from_pretrained(CryptoBERT)

#Creo pipeline "sentiment-analysis" per automatizzare il processo
sentiment = pipeline(
    "sentiment-analysis", 
    model=model, 
    tokenizer=tokenizer, 
    device=device
)

print("Modello pronto per l'analisi.")

Caricamento di ElKulako/cryptobert sulla GPU


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: ElKulako/cryptobert
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Modello pronto per l'analisi.


In [16]:
df = pd.read_csv('dataset_filtrato.csv')  #carico il dataset filtrato
testi = df['text'].astype(str).tolist()   #creo variabile per i testi dei tweet

#SETUP
batch_size = 64       #numero di tweet letti contemporaneamente
risultati_label = []  #creo lista per l'etichetta assegnata ad ogni tweet
risultati_score = []  #creo lista per lo score assegnato ad ogni tweet

print(f"Inizio analisi di {len(testi):,}") #verifico se il numero coincide con quello ottenuto precedentemente

#Creo ciclo for per leggere 64 tweet contemporaneamente
for i in tqdm(range(0, len(testi), batch_size), desc="Analisi Sentiment"):
    batch = testi[i : i + batch_size]
    
    output = sentiment(batch, truncation=True, max_length=128) #truncation=True garantisce che tweet oltre i 128 token non blocchino il processo
    
    #estraggo label e score singolarmente (*la pipeline non restituisce una lista di 64 parole ma una lista di dizionari)
    risultati_label.extend([r['label'] for r in output]) #prende solo i valori associati alla chiave label
    risultati_score.extend([r['score'] for r in output]) ##prende solo i valori associati alla chiave score

#Inserisco i risultati nel dataset
df['sentiment_label'] = risultati_label
df['sentiment_score'] = risultati_score

#Creo dataset con il sentiment (lo salvo in locale)
df.to_csv('dataset_con_sentiment.csv', index=False)

print("\nAnalisi conclusa e dati salvati in 'dataset_con_sentiment.csv'")

Inizio analisi di 510,073


Analisi Sentiment: 100%|█████████████████████████████████████████████████████████| 7970/7970 [5:00:40<00:00,  2.26s/it]



Analisi conclusa e dati salvati in 'dataset_con_sentiment.csv'


In [7]:
#Carico il dataset con i risultati di CryptoBERT
data = 'dataset_con_sentiment.csv'
df = pd.read_csv(data)

df.head(5)

,date,n_replies,n_likes,n_retweets,text,sentiment_label,sentiment_score
0,2019-05-21 16:49:45+00:00,47.0,81.0,84.0,"BTC IS STILL GOING STRONG!!\n\nThus, we are gi...",Neutral,0.548231
1,2019-05-22 12:42:16+00:00,3.0,2.0,7.0,BestMixer has been seized by the Dutch Police ...,Neutral,0.683266
2,2019-05-27 11:49:30+00:00,1.0,1.0,1.0,Invested my Life Savings into Bitcoin and Ethe...,Neutral,0.708408
3,2019-05-27 08:13:06+00:00,5.0,167.0,68.0,"Bitcoin Price Hits $8,939 in New 2019 High: Wh...",Neutral,0.884856
4,2019-05-27 11:49:17+00:00,1.0,1.0,1.0,"#Countdown #ComingSoon The $QLC airdrop, #Q_Ga...",Neutral,0.517616


In [8]:
#Controllo valori nulli
valori_nulli = df.isnull().sum()

if valori_nulli.any():
    print(valori_nulli[valori_nulli > 0])
    #rimuovo eventuali righe con valori nulli
    df = df.dropna(subset=colonne_richieste)
else:
    print("non ci sono valori nulli")

non ci sono valori nulli


In [9]:
#Controllo il tipo di dato delle colonne

#format = mixed permette di leggere sia date in diverso formato
df['date'] = pd.to_datetime(df['date'], format='mixed', errors='coerce', utc=True) #utc=true perchè presenti più fusi orari
df['sentiment_score'] = pd.to_numeric(df['sentiment_score'])
df['n_replies'] = pd.to_numeric(df['n_replies']).astype(int)
df['n_likes'] = pd.to_numeric(df['n_likes']).astype(int)
df['n_retweets'] = pd.to_numeric(df['n_retweets']).astype(int)

print(f"Tweet disponibili: {len(df):,}")
df.dtypes

Tweet disponibili: 510,073


date               datetime64[us, UTC]
n_replies                        int64
n_likes                          int64
n_retweets                       int64
text                               str
sentiment_label                    str
sentiment_score                float64
dtype: object

In [10]:
#Verifico che il filtro engagement abbia funzionato correttamente
controllo_zeri = {
    'n_likes': (df['n_likes'] == 0).sum(),
    'n_retweets': (df['n_retweets'] == 0).sum(),
    'n_replies': (df['n_replies'] == 0).sum()
}

print(controllo_zeri)

#Intervallo temporale dati
print(df['date'].min())
print(df['date'].max())

{'n_likes': np.int64(0), 'n_retweets': np.int64(0), 'n_replies': np.int64(0)}
2018-01-02 01:00:10+00:00
2023-05-28 00:00:00+00:00


In [11]:
#CALCOLO DEL SENTIMENT

# 1. ASSEGNAZIONE DEL SEGNO 
# Trasformo le categorie qualitative in un dominio numerico: bearish --> -1 (riflette sentiment negativo) / neutral --> 0 / bullish --> 1
#ho deciso di usare 0 per i valori neutral per ridurre la polarizzazione quando i tweet verranno aggregati su base giornaliera,
#riflettendo così un mercato senza una direzione chiara.
sentiment_map = {'Bullish': 1, 'Neutral': 0, 'Bearish': -1}
df['sentiment_sign'] = df['sentiment_label'].map(sentiment_map)

# 2. CALCOLO DELL'ENGAGEMENT SCORE
#Sommo like, commenti e retweet (metriche correlate).
#In seguito applico la trasformazione logaritmica (np.log) per due ragioni:
#   A) Normalizzazione: spesso pochi tweet hanno milioni di like e senza logaritmo, un singolo tweet virale oscurerebbe migliaia di altri tweet.
#   B) Utilità Marginale: la differenza tra 100 e 1000 like è più significativa che tra 100.000 e 100.900.
df['engagement_sum'] = df['n_likes'] + df['n_retweets'] + df['n_replies']
df['engagement_weight'] = np.log(df['engagement_sum'])

# 3. CALCOLO DEL WEIGHTED SENTIMENT 
# Formula utilizzata: weighted_sentiment = Sign_i * Confidence_i * Weight_i
#sentiment_sign: La direzione del sentiment (-1, 0, 1).
#sentiment_score: quanto il modello CryptoBERT è sicuro della label assegnata al tweet.
#engagement_weight: impatto mediatico del tweet.
df['weighted_sentiment'] = df['sentiment_sign'] * df['sentiment_score'] * df['engagement_weight']

#verifica valori estremi per evitare distorsioni 
print(f"  - Massimo Peso Bullish: {df['weighted_sentiment'].max()}")
print(f"  - Massimo Peso Bearish: {df['weighted_sentiment'].min()}")

print(df.head(5))

  - Massimo Peso Bullish: 8.311575448852228
  - Massimo Peso Bearish: -9.44618323475683
                       date  n_replies  n_likes  n_retweets  \
0 2019-05-21 16:49:45+00:00         47       81          84   
1 2019-05-22 12:42:16+00:00          3        2           7   
2 2019-05-27 11:49:30+00:00          1        1           1   
3 2019-05-27 08:13:06+00:00          5      167          68   
4 2019-05-27 11:49:17+00:00          1        1           1   

                                                text sentiment_label  \
0  BTC IS STILL GOING STRONG!!\n\nThus, we are gi...         Neutral   
1  BestMixer has been seized by the Dutch Police ...         Neutral   
2  Invested my Life Savings into Bitcoin and Ethe...         Neutral   
3  Bitcoin Price Hits $8,939 in New 2019 High: Wh...         Neutral   
4  #Countdown #ComingSoon The $QLC airdrop, #Q_Ga...         Neutral   

   sentiment_score  sentiment_sign  engagement_sum  engagement_weight  \
0         0.548231         

In [12]:
#AGGREGAZIONE ORARIA

# 1. UNITÀ DI ANALISI: l'aggregazione oraria è stata scelta per rappresentare la velocità di cambiamento del sentiment 
#    e la volatilità dei mercati delle criptovalute
# 2. SELEZIONE TEMPORALE (ANNO 2019): si è scelto di isolare il 2019 in quanto rappresenta il blocco di dati 
#    più solido e numericamente rilevante del dataset (circa il 90% dei tweet totali).
# 3. LIMITE AL 23 NOVEMBRE: l'analisi viene interrotta il 23/11/2019 alle 15:00 poichè
#    oltre questa data è stato rilevato un gap di 920 ore consecutive (38 giorni). 
# 4. AGGREGAZIONE PER MEDIA: La media pesata riflette il 'sentiment prevalente'. 
#    Gli 0 dei 'Neutral' stabilizzano il segnale, evitando distorsioni da volumi elevati ma a bassa convinzione.

#Filtro per l'intervallo temporale d'interesse
df_2019 = df[df['date'].dt.year == 2019].copy()

#tengo solo i dati dove la sequenza è continua e affidabile
data_limite = '2019-11-23 16:00:00'
df_2019 = df_2019[df_2019['date'] < data_limite]

#Creo colonna oraria
df_2019['hour'] = df_2019['date'].dt.floor('h')

#Aggrego i dati calcolando il sentiment medio, il volume di tweet e l'engagement per ogni ora
hourly_data = df_2019.groupby('hour').agg({
    'weighted_sentiment': 'mean',    #mood medio pesato dell'ora
    'text': 'count',                 #numero di tweet nell'ora
    'engagement_sum': 'sum'          #risonanza totale dei tweet
}).rename(columns={
    'weighted_sentiment': 'avg_weighted_sentiment',
    'engagement_sum': 'total_engagement'
})


#Controllo filtro
prima_ora = hourly_data.index.min()
ultima_ora = hourly_data.index.max()
totale_ore = len(hourly_data)

print(f"dal {prima_ora} al {ultima_ora}")
print(f"Numero di campioni orari: {totale_ore}")

#statistica descrittiva del sentiment nel periodo selezionato
print(hourly_data['avg_weighted_sentiment'].describe())

dal 2019-01-01 00:00:00+00:00 al 2019-11-23 15:00:00+00:00
Numero di campioni orari: 7652
count    7652.000000
mean        0.606911
std         0.611481
min        -6.355293
25%         0.372005
50%         0.531101
75%         0.706971
max         6.889465
Name: avg_weighted_sentiment, dtype: float64


In [13]:
#Verifico se ci sono vuoti nella sequenza oraria

#creo calendario fittizio per confronto
calendario_teorico = pd.date_range(
    start=hourly_data.index.min(), 
    end=hourly_data.index.max(), 
    freq='h'
)

#aggiungo fuso orario UTC
if hourly_data.index.tz is not None:
    calendario_teorico = calendario_teorico.tz_convert('UTC')

#cerco ore mancanti
ore_mancanti = calendario_teorico.difference(hourly_data.index)

#verifico vuoti
print(f"Ore totali attese: {len(calendario_teorico)}")
print(f"Ore effettive con tweet: {len(hourly_data)}")
print(f"Ore mancanti (vuoti): {len(ore_mancanti)}")

Ore totali attese: 7840
Ore effettive con tweet: 7652
Ore mancanti (vuoti): 188


In [14]:
#calcolo la differenza di tempo tra un'ora mancante e la precedente
#(se la differenza è di 1 ora, significa che sono consecutive)
differenze = pd.Series(ore_mancanti).diff()

#conto quante volte le ore sono staccate (differenza > 1 ora)
nuovo_blocco = differenze != pd.Timedelta(hours=1)
id_blocchi = nuovo_blocco.cumsum()

#trovo la dimensione del blocco più grande
max_gap = pd.Series(ore_mancanti).groupby(id_blocchi).count().max()

print(f"Il massimo numero di ore consecutive mancanti è: {max_gap}")

Il massimo numero di ore consecutive mancanti è: 3


In [ ]:
#IMPUTAZIONE DEI DATI MANCANTI

#espando il dataset per includere tutte le ore del calendario (comprese le 188 ore mancanti che saranno inizialmente NaN)
hourly_data_fixed = hourly_data.reindex(calendario_teorico)

#Applicazione dell'interpolazione lineare per il Sentiment
#dato che il gap massimo è di sole 3 ore, l'assunzione di un cambiamento lineare può essere giustificata
#e preserva il trend locale.
hourly_data_fixed['avg_weighted_sentiment'] = hourly_data_fixed['avg_weighted_sentiment'].interpolate(method='linear')

#Gestione Volumi ed Engagement (Zero-Filling)
#a differenza del sentiment,se non sono presenti tweet in un'ora, l'engagement e il volume saranno pari a 0.
hourly_data_fixed['text'] = hourly_data_fixed['text'].fillna(0)
hourly_data_fixed['total_engagement'] = hourly_data_fixed['total_engagement'].fillna(0)

#verifico se sono ancora presenti valori nulli
print(f"Righe totali: {len(hourly_data_fixed)}")
print(f"Valori nulli residui: {hourly_data_fixed.isnull().sum().sum()}")

hourly_data_fixed.head(5)

Righe totali: 7840
Valori nulli residui: 0


,avg_weighted_sentiment,text,total_engagement
2019-01-01 00:00:00+00:00,0.000000,1.0,1071.0
2019-01-01 01:00:00+00:00,1.639163,2.0,520.0
2019-01-01 02:00:00+00:00,0.000000,1.0,19.0
2019-01-01 03:00:00+00:00,4.123879,1.0,1056.0
2019-01-01 04:00:00+00:00,3.372124,0.0,0.0


In [16]:
#manca nome colonna date (ritenuta come indice) ed è presente il fuso orario

#rimozione fuso orario
hourly_data_fixed.index = hourly_data_fixed.index.tz_localize(None)

#trasformo l'indice in una colonna e rinomino le colonne
hourly_data_fixed = hourly_data_fixed.reset_index().rename(columns={
    'index': 'date',
    'text': 'tweet_volume'
})

print(hourly_data_fixed.head(5))

                 date  avg_weighted_sentiment  tweet_volume  total_engagement
0 2019-01-01 00:00:00                0.000000           1.0            1071.0
1 2019-01-01 01:00:00                1.639163           2.0             520.0
2 2019-01-01 02:00:00                0.000000           1.0              19.0
3 2019-01-01 03:00:00                4.123879           1.0            1056.0
4 2019-01-01 04:00:00                3.372124           0.0               0.0


In [17]:
#esporto il dataset
hourly_data_fixed.to_csv('sentiment_2019_orario.csv', index=False)

PROCEDO ALL'UNIONE DI TUTTI I DATASET CREATI

In [18]:
sentiment = pd.read_csv('sentiment_2019_orario.csv')
index = pd.read_csv('fear_and_greed_index.csv')
bitcoin = pd.read_csv('btcusd_2019_orario.csv')

print(sentiment.head(5))
print(index.head(5))
print(bitcoin.head(5))

print(sentiment.tail(5))
print(index.tail(5))
print(bitcoin.tail(5))

                  date  avg_weighted_sentiment  tweet_volume  total_engagement
0  2019-01-01 00:00:00                0.000000           1.0            1071.0
1  2019-01-01 01:00:00                1.639163           2.0             520.0
2  2019-01-01 02:00:00                0.000000           1.0              19.0
3  2019-01-01 03:00:00                4.123879           1.0            1056.0
4  2019-01-01 04:00:00                3.372124           0.0               0.0
    fng_value  fng_classification          date
0  07-09-2022                  24  Extreme Fear
1  06-09-2022                  22  Extreme Fear
2  05-09-2022                  23  Extreme Fear
3  04-09-2022                  20  Extreme Fear
4  03-09-2022                  21  Extreme Fear
                  date     Open     High      Low    Close      Volume
0  2019-01-01 00:00:00  3750.62  3752.01  3630.00  3690.91  603.089248
1  2019-01-01 01:00:00  3685.65  3690.78  3663.13  3681.57  142.961312
2  2019-01-01 02:00:00  3

I dataset "bitcoin" e "sentiment" sono equivalenti nella colonna date (data di inizio, data di fine e composizione su base oraria), mentre il dataset "index" è su base giornaliera. Questo dataset contiene dati storici completi raccolti dal sito web Alternative.me sul Crypto Fear and Greed Index). Il Crypto Fear and Greed Index è una metrica popolare utilizzata per misurare le emozioni e il sentiment del mercato delle criptovalute, fornendo approfondimenti sul comportamento del mercato e sui potenziali movimenti futuri. Come è composto? Per rispondere, verranno inserite integralmente le informazioni contenute nel sito ufficiale: https://alternative.me/crypto/fear-and-greed-index/

Data Sources We are gathering data from the five following sources. Each data point is valued the same as the day before in order to visualize a meaningful progress in sentiment change of the crypto market. First of all, the current index is for bitcoin only (we offer separate indices for large alt coins soon), because a big part of it is the volatility of the coin price. But let’s list all the different factors we’re including in the current index:

Volatility (25 %) We’re measuring the current volatility and max. drawdowns of bitcoin and compare it with the corresponding average values of the last 30 days and 90 days. We argue that an unusual rise in volatility is a sign of a fearful market.

Market Momentum/Volume (25%) Also, we’re measuring the current volume and market momentum (again in comparison with the last 30/90 day average values) and put those two values together. Generally, when we see high buying volumes in a positive market on a daily basis, we conclude that the market acts overly greedy / too bullish.

Social Media (15%) While our reddit sentiment analysis is still not in the live index (we’re still experimenting some market-related key words in the text processing algorithm), our twitter analysis is running. There, we gather and count posts on various hashtags for each coin (publicly, we show only those for Bitcoin) and check how fast and how many interactions they receive in certain time frames). A unusual high interaction rate results in a grown public interest in the coin and in our eyes, corresponds to a greedy market behaviour.

Surveys (15%) currently paused Together with strawpoll.com (disclaimer: we own this site, too), quite a large public polling platform, we’re conducting weekly crypto polls and ask people how they see the market. Usually, we’re seeing 2,000 - 3,000 votes on each poll, so we do get a picture of the sentiment of a group of crypto investors. We don’t give those results too much attention, but it was quite useful in the beginning of our studies. You can see some recent results here.

Dominance (10%) The dominance of a coin resembles the market cap share of the whole crypto market. Especially for Bitcoin, we think that a rise in Bitcoin dominance is caused by a fear of (and thus a reduction of) too speculative alt-coin investments, since Bitcoin is becoming more and more the safe haven of crypto. On the other side, when Bitcoin dominance shrinks, people are getting more greedy by investing in more risky alt-coins, dreaming of their chance in next big bull run. Anyhow, analyzing the dominance for a coin other than Bitcoin, you could argue the other way round, since more interest in an alt-coin may conclude a bullish/greedy behaviour for that specific coin.

Trends (10%) We pull Google Trends data for various Bitcoin related search queries and crunch those numbers, especially the change of search volumes as well as recommended other currently popular searches. For example, if you check Google Trends for "Bitcoin", you can’t get much information from the search volume. But currently, you can see that there is currently a +1,550% rise of the query „bitcoin price manipulation“ in the box of related search queries (as of 05/29/2018). This is clearly a sign of fear in the market, and we use that for our index.

In [19]:
#Poichè l'informazione contenuta nell'indice è su base giornaliera, il dato verrà aggiunto ad ognuna delle 24 righe 
#rappresentanti le ore dello stesso giorno.

#*****ERRORE NEL NOME COLONNE: le colonne 'date','fng_value' e 'fng_classification' sono invertite
#*****il formato non è datetime(AAAA-MM-GG) ma GG-MM-AAAA
#*****i giorni vanno dal più recente al meno recente


#rinomino le colonne scambiamo i nomi
index = index.rename(columns={'fng_value': 'date', 'date': 'fng_value'})

#converto in formato datetime
index['date'] = pd.to_datetime(index['date'], dayfirst=True)

#taglio il dataset per corrispondenza su data inizio/fine
inizio = '2019-01-01'
fine = '2019-11-23'


index['date'] = pd.to_datetime(index['date'])
taglio = (index['date'] >= inizio) & (index['date'] <= fine)
df_index = index.loc[taglio].sort_values(by = 'date', ascending = True).reset_index(drop=True)

print(df_index.head(5))
print(df_index.tail(5))

        date  fng_classification     fng_value
0 2019-01-01                  24  Extreme Fear
1 2019-01-02                  30          Fear
2 2019-01-03                  33          Fear
3 2019-01-04                  48       Neutral
4 2019-01-05                  36          Fear
          date  fng_classification     fng_value
322 2019-11-19                  32          Fear
323 2019-11-20                  32          Fear
324 2019-11-21                  30          Fear
325 2019-11-22                  20  Extreme Fear
326 2019-11-23                  23  Extreme Fear


In [20]:
df_orario = pd.merge(sentiment, bitcoin, on='date') #.merge per unire i dataset grazie alla colonna in comune
print(df_orario.head(5))
print(df_orario.tail(5))

                  date  avg_weighted_sentiment  tweet_volume  \
0  2019-01-01 00:00:00                0.000000           1.0   
1  2019-01-01 01:00:00                1.639163           2.0   
2  2019-01-01 02:00:00                0.000000           1.0   
3  2019-01-01 03:00:00                4.123879           1.0   
4  2019-01-01 04:00:00                3.372124           0.0   

   total_engagement     Open     High      Low    Close      Volume  
0            1071.0  3750.62  3752.01  3630.00  3690.91  603.089248  
1             520.0  3685.65  3690.78  3663.13  3681.57  142.961312  
2              19.0  3681.00  3685.74  3660.00  3678.00  175.021283  
3            1056.0  3676.06  3686.45  3656.60  3677.91  196.853708  
4               0.0  3684.68  3697.35  3679.98  3693.30  169.208314  
                     date  avg_weighted_sentiment  tweet_volume  \
7835  2019-11-23 11:00:00                0.880034          35.0   
7836  2019-11-23 12:00:00                0.463974          47

In [21]:
#per sicurezza rendo datetime la colonna date
df_orario['date'] = pd.to_datetime(df_orario['date'])

#creo una colonna temporanea 'solo_data' (escludendo la parte oraria) per il match con il dataset dell'indice (che non ha informazione oraria)
df_orario['solo_data'] = df_orario['date'].dt.date
df_orario.head(5)

,date,avg_weighted_sentiment,tweet_volume,total_engagement,Open,High,Low,Close,Volume,solo_data
0,2019-01-01 00:00:00,0.000000,1.0,1071.0,3750.62,3752.01,3630.00,3690.91,603.089248,2019-01-01
1,2019-01-01 01:00:00,1.639163,2.0,520.0,3685.65,3690.78,3663.13,3681.57,142.961312,2019-01-01
2,2019-01-01 02:00:00,0.000000,1.0,19.0,3681.00,3685.74,3660.00,3678.00,175.021283,2019-01-01
3,2019-01-01 03:00:00,4.123879,1.0,1056.0,3676.06,3686.45,3656.60,3677.91,196.853708,2019-01-01
4,2019-01-01 04:00:00,3.372124,0.0,0.0,3684.68,3697.35,3679.98,3693.30,169.208314,2019-01-01


In [22]:
#creo colonna temporanea 'solo_data' anche per l'altro dataset (df_index)
df_index['solo_data'] = df_index['date'].dt.date
df_index.head(5)

,date,fng_classification,fng_value,solo_data
0,2019-01-01,24,Extreme Fear,2019-01-01
1,2019-01-02,30,Fear,2019-01-02
2,2019-01-03,33,Fear,2019-01-03
3,2019-01-04,48,Neutral,2019-01-04
4,2019-01-05,36,Fear,2019-01-05


In [23]:
#unisco nel dataset finale tramite la colonna temporanea 'solo_data'
df_knime = pd.merge(df_orario, df_index, on='solo_data')

#rimuovo la colonna temporanea
df_knime = df_knime.drop(columns=['solo_data'])

print(df_knime.head(5))
print(df_knime.tail(5))

               date_x  avg_weighted_sentiment  tweet_volume  total_engagement  \
0 2019-01-01 00:00:00                0.000000           1.0            1071.0   
1 2019-01-01 01:00:00                1.639163           2.0             520.0   
2 2019-01-01 02:00:00                0.000000           1.0              19.0   
3 2019-01-01 03:00:00                4.123879           1.0            1056.0   
4 2019-01-01 04:00:00                3.372124           0.0               0.0   

      Open     High      Low    Close      Volume     date_y  \
0  3750.62  3752.01  3630.00  3690.91  603.089248 2019-01-01   
1  3685.65  3690.78  3663.13  3681.57  142.961312 2019-01-01   
2  3681.00  3685.74  3660.00  3678.00  175.021283 2019-01-01   
3  3676.06  3686.45  3656.60  3677.91  196.853708 2019-01-01   
4  3684.68  3697.35  3679.98  3693.30  169.208314 2019-01-01   

   fng_classification     fng_value  
0                  24  Extreme Fear  
1                  24  Extreme Fear  
2             

In [24]:
#pulizia colonne
df_knime = df_knime.drop(columns=['date_y'])
#rinomino colonne
df_knime = df_knime.rename(columns={'date_x': 'date', 'fng_classification': 'F&G_value', 'fng_value': 'F&G_classification'})

df_knime.head(5)   

,date,avg_weighted_sentiment,tweet_volume,total_engagement,Open,High,Low,Close,Volume,F&G_value,F&G_classification
0,2019-01-01 00:00:00,0.000000,1.0,1071.0,3750.62,3752.01,3630.00,3690.91,603.089248,24,Extreme Fear
1,2019-01-01 01:00:00,1.639163,2.0,520.0,3685.65,3690.78,3663.13,3681.57,142.961312,24,Extreme Fear
2,2019-01-01 02:00:00,0.000000,1.0,19.0,3681.00,3685.74,3660.00,3678.00,175.021283,24,Extreme Fear
3,2019-01-01 03:00:00,4.123879,1.0,1056.0,3676.06,3686.45,3656.60,3677.91,196.853708,24,Extreme Fear
4,2019-01-01 04:00:00,3.372124,0.0,0.0,3684.68,3697.35,3679.98,3693.30,169.208314,24,Extreme Fear


Per catturare la dinamica intra-oraria del mercato ed evitare la perdita di informazioni intrinseca nell'uso del solo prezzo di chiusura, introduco la volatilità di Garman-Klass. Questo stimatore sfrutta tutti e quattro i dati OHLC (Apertura, Massimo, Minimo e Chiusura) per valutare la varianza del prezzo all'interno di ciascun intervallo temporale. L'integrazione di questa metrica fornisce alla rete neurale un segnale esplicito sui regimi di incertezza e agitazione del mercato, migliorando la capacità del modello di contestualizzare le variazioni di prezzo e i picchi nelle metriche di sentiment social.

In [ ]:
#Calcolo i rapporti logaritmici

#Logaritmo naturale del rapporto tra il prezzo Massimo (High) e il prezzo Minimo (Low) dell'ora per misurare l'ampiezza delle oscillazioni 
#avvenuta nella singola ora. Il logaritmo serve a rendere le variazioni percentuali simmetriche e indipendenti dalla scala del prezzo.
log_hl = np.log(df_knime["High"] / df_knime["Low"])

#Logaritmo naturale del rapporto tra il prezzo di Chiusura (Close) e il prezzo di Apertura (Open). Questo valore cattura il rendimento netto dell'ora.
log_co = np.log(df_knime["Close"] / df_knime["Open"])


#Formula di Garman-Klass
#definisco la costante teorica della formula di Garman-Klass: (2 * ln(2) - 1) ≈ 0.386294.
#Questa costante deriva dalla teoria dei moti browniani e serve a pesare correttamente la varianza
#tra il prezzo d'apertura/chiusura e massimo/minimo.
costante_gk = 2 * np.log(2) - 1

#calcolo la varianza di Garman-Klass secondo la formula teorica:
# 0.5 * [ln(High/Low)]^2 - (2*ln(2) - 1) * [ln(Close/Open)]^2
varianza = 0.5 * (log_hl**2) - costante_gk * (log_co**2)

#Controllo di sicurezza:
#inserisco np.maximum(0, varianza) in modo tale che se per imprecisioni di calcolo sui decimali
#la varianza risulta leggermente negativa, viene forzata a 0, evitando errori di radici di numeri negativi.
df_knime["vol_garman_klass"] = np.sqrt(np.maximum(0, varianza))

df_knime.head(5)

,date,avg_weighted_sentiment,tweet_volume,total_engagement,Open,High,Low,Close,Volume,F&G_value,F&G_classification,vol_garman_klass
0,2019-01-01 00:00:00,0.000000,1.0,1071.0,3750.62,3752.01,3630.00,3690.91,603.089248,24,Extreme Fear,0.021142
1,2019-01-01 01:00:00,1.639163,2.0,520.0,3685.65,3690.78,3663.13,3681.57,142.961312,24,Extreme Fear,0.005273
2,2019-01-01 02:00:00,0.000000,1.0,19.0,3681.00,3685.74,3660.00,3678.00,175.021283,24,Extreme Fear,0.004930
3,2019-01-01 03:00:00,4.123879,1.0,1056.0,3676.06,3686.45,3656.60,3677.91,196.853708,24,Extreme Fear,0.005740
4,2019-01-01 04:00:00,3.372124,0.0,0.0,3684.68,3697.35,3679.98,3693.30,169.208314,24,Extreme Fear,0.002996


In [26]:
#Rimozione variabili non informative
#rimuovo le informazioni riguardo i prezzi Open, High e Low (già sintetizzati nella volatilità di Garman-Klass e fortemente correlati a prezzi Close)
#rimuovo la colonna "F&G_classification" --> categorizzazione del valore numerico contenuto nella colonna "F&G_value"
df_knime = df_knime.drop(columns=["Open", "High", "Low", "F&G_classification"])

df_knime.head(5)

,date,avg_weighted_sentiment,tweet_volume,total_engagement,Close,Volume,F&G_value,vol_garman_klass
0,2019-01-01 00:00:00,0.000000,1.0,1071.0,3690.91,603.089248,24,0.021142
1,2019-01-01 01:00:00,1.639163,2.0,520.0,3681.57,142.961312,24,0.005273
2,2019-01-01 02:00:00,0.000000,1.0,19.0,3678.00,175.021283,24,0.004930
3,2019-01-01 03:00:00,4.123879,1.0,1056.0,3677.91,196.853708,24,0.005740
4,2019-01-01 04:00:00,3.372124,0.0,0.0,3693.30,169.208314,24,0.002996


In [ ]:
#ESTRAGGO IL DATASET FINALE
df_knime.to_csv('dataset_knime.csv', index=False)